# K2: BoW + stablo odlučivanja (komentari mobilnih aplikacija)
**Tip:** Blok 1. Reprezentacija: Bag of Words (`CountVectorizer`). Klasifikator: stablo odlučivanja. Uz tekst idu i numeričke kolone.

### Zadatak
U fajlu `mobile_apps_reviews.csv` su komentari korisnika mobilnih aplikacija sa ručno označenom kategorijom (`category`).
- Od kolone `category` napraviti izlaznu varijablu `cls` sa tri vrednosti: **funkcionalni** (BUG, FEATURE), **nefunkcionalni** (PERFORMANCE, USABILITY, ENERGY, SECURITY), **ostalo** (OTHER).
- Transformisati kolonu `review` primenom **Bag of Words modela (TF metrika, CountVectorizer)**. Proceniti koje još varijable uključiti i obrazložiti.
- Kreirati prvi model primenom **stabla odlučivanja** sa podrazumevanim hiperparametrima.
- Izračunati preciznost, odziv, F1 po klasama i ukupno, i balanced accuracy. Interpretirati.
- Drugi model podešavanjem hiperparametara: komentar sa opsezima, pa jedan hiperparametar za BoW i jedan za stablo kroz 5-fold kros-validaciju.
- Metrike za podešeni model i poređenje modela.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, izbačeni komentari bez teksta i duplikati teksta.
- Napravljena izlazna varijabla `cls` (3 klase, neravnomerne).
- Dodata kolona `review_len` (broj reči). Pregledane `review_len` (kdeplot) i `rating` (boxplot), i broj vrednosti za `app`, `version`, `device`.
- Odluka: ulaz su `review`, `review_len` i `rating` (raspodele se razlikuju po klasama). `id`, `user`, `date` nisu bitni, a `app`, `version`, `device` imaju previše vrednosti.
- X je DataFrame sa 3 kolone, pa ti treba `ColumnTransformer`. Podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'mobile_apps_reviews.csv')
data.info()

In [ ]:
data = data.dropna(subset=['review'])
print('Duplikati teksta:', data.duplicated(subset='review').sum())
data = data.drop_duplicates(subset='review').reset_index(drop=True)
print(data['category'].value_counts())

In [ ]:
def category_to_cls(cat):
    if cat == 'OTHER':
        return 'ostalo'
    return 'funkcionalni' if cat in ['BUG', 'FEATURE'] else 'nefunkcionalni'

data['cls'] = data['category'].apply(category_to_cls)
print(data['cls'].value_counts(normalize=True))

In [ ]:
data['review_len'] = data['review'].str.split().str.len()

plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='review_len', hue='cls',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Duzina komentara po klasama')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

plt.figure(figsize=(8, 6))
sb.boxplot(data=data, x='cls', y='rating', hue='cls', fill=True, saturation=0.75,
           width=0.55, gap=0.25, linewidth=2, legend=False)
plt.title('Raspodela ocena po klasama')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print(data[['app', 'version', 'device']].nunique())

In [ ]:
X = data[['review', 'review_len', 'rating']]
y = data['cls']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. spaCy `en_core_web_sm` i funkcija `spacy_srediTekst`
2. `CountVectorizer(tokenizer=spacy_srediTekst, token_pattern=None, min_df=0.01)`
3. `ColumnTransformer`: BoW na `review`, a `review_len` i `rating` kroz `remainder='passthrough'` (stablu ne treba skaliranje)
4. `DecisionTreeClassifier(random_state=RAND_STATE)` i `Pipeline`, pa `fit`
5. Matrica konfuzije, `classification_report`, balanced accuracy, tumačenje
6. Komentar o hiperparametrima, pa `GridSearchCV` (npr. `min_df` + `max_depth`). Pazi na ime: korak__deo__parametar
7. Podešeni model i poređenje

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }